In [28]:
import pandas as pd


df = pd.read_parquet('../Imdb_Movie_Dataset.parquet')
df_aux = df.copy()

df.head(10)

,id,title,vote_average,vote_count,status,release_date,revenue,runtime,adult,budget,...,original_language,original_title,overview,popularity,tagline,genres,production_companies,production_countries,spoken_languages,keywords
0,27205,Inception,8.364,34495,Released,7/15/2010,825532764,148,False,160000000,...,en,Inception,"Cobb, a skilled thief who commits corporate es...",83.952,Your mind is the scene of the crime.,"Action, Science Fiction, Adventure","Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America","English, French, Japanese, Swahili","rescue, mission, dream, airplane, paris, franc..."
1,157336,Interstellar,8.417,32571,Released,11/5/2014,701729206,169,False,165000000,...,en,Interstellar,The adventures of a group of explorers who mak...,140.241,Mankind was born on Earth. It was never meant ...,"Adventure, Drama, Science Fiction","Legendary Pictures, Syncopy, Lynda Obst Produc...","United Kingdom, United States of America",English,"rescue, future, spacecraft, race against time,..."
2,155,The Dark Knight,8.512,30619,Released,7/16/2008,1004558444,152,False,185000000,...,en,The Dark Knight,Batman raises the stakes in his war on crime. ...,130.643,Welcome to a world without rules.,"Drama, Action, Crime, Thriller","DC Comics, Legendary Pictures, Syncopy, Isobel...","United Kingdom, United States of America","English, Mandarin","joker, sadism, chaos, secret identity, crime f..."
3,19995,Avatar,7.573,29815,Released,12/15/2009,2923706026,162,False,237000000,...,en,Avatar,"In the 22nd century, a paraplegic Marine is di...",79.932,Enter the world of Pandora.,"Action, Adventure, Fantasy, Science Fiction","Dune Entertainment, Lightstorm Entertainment, ...","United States of America, United Kingdom","English, Spanish","future, society, culture clash, space travel, ..."
4,24428,The Avengers,7.710,29166,Released,4/25/2012,1518815515,143,False,220000000,...,en,The Avengers,When an unexpected enemy emerges and threatens...,98.082,Some assembly required.,"Science Fiction, Action, Adventure",Marvel Studios,United States of America,"English, Hindi, Russian","new york city, superhero, shield, based on com..."
5,293660,Deadpool,7.606,28894,Released,2/9/2016,783100000,108,False,58000000,...,en,Deadpool,The origin story of former Special Forces oper...,72.735,Witness the beginning of a happy ending.,"Action, Adventure, Comedy","20th Century Fox, The Donners' Company, Genre ...",United States of America,English,"superhero, anti hero, mercenary, based on comi..."
6,299536,Avengers: Infinity War,8.255,27713,Released,4/25/2018,2052415039,149,False,300000000,...,en,Avengers: Infinity War,As the Avengers and their allies have continue...,154.340,An entire universe. Once and for all.,"Adventure, Action, Science Fiction",Marvel Studios,United States of America,"English, Xhosa","sacrifice, magic, superhero, based on comic, s..."
7,550,Fight Club,8.438,27238,Released,10/15/1999,100853753,139,False,63000000,...,en,Fight Club,A ticking-time-bomb insomniac and a slippery s...,69.498,Mischief. Mayhem. Soap.,Drama,"Regency Enterprises, Fox 2000 Pictures, Taurus...",United States of America,English,"dual identity, rage and hate, based on novel o..."
8,118340,Guardians of the Galaxy,7.906,26638,Released,7/30/2014,772776600,121,False,170000000,...,en,Guardians of the Galaxy,"Light years from Earth, 26 years after being a...",33.255,All heroes start somewhere.,"Action, Science Fiction, Adventure",Marvel Studios,United States of America,English,"spacecraft, based on comic, space, orphan, adv..."
9,680,Pulp Fiction,8.488,25893,Released,9/10/1994,213900000,154,False,8500000,...,en,Pulp Fiction,"A burger-loving hit man, his philosophical par...",74.862,Just because you are a character doesn't mean ...,"Thriller, Crime","Miramax, A Band Apart, Jersey Films",United States of America,"English, Spanish, French","drug dealer, boxer, massage, stolen money, bri..."


In [29]:
# Exibe os nomes e tipos de dados das colunas da tabela
df.dtypes

id                        int64
title                       str
vote_average            float64
vote_count                int64
status                      str
release_date                str
revenue                   int64
runtime                   int64
adult                      bool
budget                    int64
imdb_id                     str
original_language           str
original_title              str
overview                    str
popularity              float64
tagline                     str
genres                      str
production_companies        str
production_countries        str
spoken_languages            str
keywords                    str
dtype: object

**Cluterização**
---

In [30]:
import pandas as pd
import numpy as np

"""
Estratégia de Preparação de Dados para Clusterização K-Means
----------------------------------------------------------------------
Objetivo: Identificar perfis de filmes (blockbusters, nichos e clássicos) evitando vieses de dados ausentes.

Decisões Metodológicas:
1. Relevância Estatística: Filtro de 'vote_count >= 50' para garantir que a 'vote_average' seja 
   estatisticamente confiável e evitar ruído de filmes com apenas um ou dois votos.
2. Imputação por Mediana: Substituímos valores nulos/zeros em 'runtime' e 'budget' pela mediana 
   para preservar filmes independentes que possuem metadados incompletos, atendendo à premissa 
   de não descartar registros importantes.
3. Transformação Logarítmica: Aplicada em variáveis de alta variância (Popularidade, Votos e Orçamento) 
   para reduzir o impacto de outliers (blockbusters) e permitir que o K-Means identifique padrões 
   em filmes de menor escala.
4. Contexto Temporal: Inclusão do 'release_year' para agrupar filmes por épocas de lançamento.
"""

# 1. Tratamento de Data e Seleção de Features
df_aux['release_year'] = pd.to_datetime(df_aux['release_date'], errors='coerce').dt.year
features = ['popularity', 'vote_average', 'vote_count', 'runtime', 'budget', 'release_year']
df_clustering = df_aux[features].copy()

# 2. Limpeza de 'Ruído' (Filtro de Relevância)
# Mantemos apenas quem tem um mínimo de votos antes de preencher os nulos
df_clustering = df_clustering[df_clustering['vote_count'] >= 50]

# 3. Imputação pela Mediana (Tratando Zeros e NaNs)
cols_to_fix = ['runtime', 'budget', 'release_year']
for col in cols_to_fix:
    df_clustering[col] = df_clustering[col].replace(0, np.nan)
    df_clustering[col] = df_clustering[col].fillna(df_clustering[col].median())

# 4. Normalização da Distribuição (Log Transformation)
# O log1p é usado para evitar erro de log(0)
df_clustering['popularity'] = np.log1p(df_clustering['popularity'])
df_clustering['vote_count'] = np.log1p(df_clustering['vote_count'])
df_clustering['budget'] = np.log1p(df_clustering['budget'])

print(f'Total de registros para clusterização: {len(df_clustering)}')
display(df_clustering.head())

Total de registros para clusterização: 27968


,popularity,vote_average,vote_count,runtime,budget,release_year
0,4.442086,8.364,10.448599,148.0,18.890684,2010.0
1,4.950468,8.417,10.391208,169.0,18.921456,2014.0
2,4.880094,8.512,10.329409,152.0,19.035866,2008.0
3,4.393609,7.573,10.302800,162.0,19.283571,2009.0
4,4.595948,7.710,10.280793,143.0,19.209138,2012.0


In [31]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_clustering)

print('Dados escalonados com sucesso.')

Dados escalonados com sucesso.


In [32]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)

df_result = df_aux.loc[df_clustering.index].copy()
df_result['cluster'] = clusters

print('Distribuição por Cluster:')
print(df_result['cluster'].value_counts())

Distribuição por Cluster:
cluster
2    9031
1    8693
0    5975
3    4269
Name: count, dtype: int64


In [33]:
# No seu arquivo do Dash (.py), atualize a função de predição:

def predict_cluster(popularity, vote_average, vote_count, runtime, budget, release_year):
    # 1. Organiza os dados na ordem exata que treinamos no Notebook
    dados = pd.DataFrame([[
        popularity, 
        vote_average, 
        vote_count, 
        runtime, 
        budget, 
        release_year
    ]], columns=['popularity', 'vote_average', 'vote_count', 'runtime', 'budget', 'release_year'])
    
    # 2. Aplica o Log Transformation (Obrigatório, pois o modelo foi treinado assim)
    dados['popularity'] = np.log1p(dados['popularity'])
    dados['vote_count'] = np.log1p(dados['vote_count'])
    dados['budget'] = np.log1p(dados['budget'])
    
    # 3. Escala e Prediz
    dados_scaled = scaler.transform(dados)
    cluster = kmeans.predict(dados_scaled)
    
    return int(cluster)

In [34]:
print('\n--- Perfil dos Clusters (Médias Reais) ---')
# Agrupamos pelas médias para entender quem é quem
analise_clusters = df_result.groupby('cluster')[features].mean()
display(analise_clusters)

# Opcional: Ver exemplo de filmes no Cluster com menos itens (geralmente os blockbusters)
menor_cluster = df_result['cluster'].value_counts().idxmin()
print(f'\nExemplos de filmes no Cluster {menor_cluster}:')
display(df_result[df_result['cluster'] == menor_cluster][['title', 'vote_average', 'popularity']].head(10))


--- Perfil dos Clusters (Médias Reais) ---


,popularity,vote_average,vote_count,runtime,budget,release_year
cluster,,,,,,
0,34.464809,6.718743,2616.049707,109.607197,3.394716e+07,2006.907448
1,8.939849,5.483101,164.927413,89.874152,3.068524e+06,2007.992365
2,9.080789,6.908521,187.271177,104.542354,1.970391e+06,2009.974972
3,9.552769,6.803882,214.541579,93.003514,6.820922e+05,1961.858515



Exemplos de filmes no Cluster 3:


,title,vote_average,popularity
514,Snow White and the Seven Dwarfs,7.116,53.193
656,Rear Window,8.359,27.612
744,Pinocchio,7.102,34.383
789,Bambi,7.000,43.491
812,Dr. Strangelove or: How I Learned to Stop Worr...,8.129,18.253
832,Citizen Kane,8.015,28.218
852,Casablanca,8.171,25.177
922,Dumbo,6.995,60.157
1120,It's a Wonderful Life,8.260,26.492
1162,A Fistful of Dollars,7.848,29.315


In [35]:
cluster_3_movies = df_result[df_result['cluster'] == 3]
display(cluster_3_movies[['title', 'popularity', 'vote_average', 'vote_count', 'runtime']])

,title,popularity,vote_average,vote_count,runtime
514,Snow White and the Seven Dwarfs,53.193,7.116,6840,83
656,Rear Window,27.612,8.359,5893,112
744,Pinocchio,34.383,7.102,5440,88
789,Bambi,43.491,7.000,5236,70
812,Dr. Strangelove or: How I Learned to Stop Worr...,18.253,8.129,5111,95
...,...,...,...,...,...
27947,"What Happened on Twenty-Third Street, New York...",1.884,5.580,50,1
27952,Night of the Demon,6.053,4.880,50,96
27962,It Always Rains on Sunday,5.824,6.390,50,92
27964,It Can't Be!,4.886,7.390,50,92


In [36]:
import joblib
import os

os.makedirs('models', exist_ok=True)

joblib.dump(scaler, 'models/scaler_cluster.pkl')
joblib.dump(kmeans, 'models/kmeans_model.pkl')

print("Modelos salvos com sucesso na pasta 'models/'!")

Modelos salvos com sucesso na pasta 'models/'!
